# Agentic work - step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will demonstrate an agent with real function calling that reads machine state, decides to assign work, and remembers what it did in a table. On the second call, the agent refuses to double-assign based on memory.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the factory is loaded on your branch.

## Setup - initialize

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, _env, client

NAMESPACE = "default"
SESSION_ID = "shift-001"
WO_ID = "WO-1001"
MACHINE_ID = "M-204"
TABLE_MEMORY = "factory.agent_memory"
TABLE_MACHINE = "factory.machine"
TABLE_OPERATION = "factory.operation"

print(f'Work order: {WO_ID}, Machine: {MACHINE_ID}')

## Step 1 - Show initial state

Check the machine status and work order status. The SQL:

```sql
SELECT machine_id, status, health_pct FROM factory.machine WHERE machine_id='M-204';
SELECT wo_id, product_id, qty, status FROM factory.work_order WHERE wo_id='WO-1001';
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT machine_id, status, health_pct FROM {TABLE_MACHINE} WHERE machine_id=%s", [MACHINE_ID])
        mach = cur.fetchone()
        cur.execute(f"SELECT wo_id, product_id, qty, status FROM factory.work_order WHERE wo_id=%s", [WO_ID])
        wo = cur.fetchone()

print(f'Initial state:')
print(f'  {MACHINE_ID}: {mach[1]}, {mach[2]}%')
print(f'  {WO_ID}: {wo[3]}, {wo[2]} units')

## Step 2 - Agent's function call 1: read available machines

The agent queries for machines that are healthy and ready for work. The SQL:

```sql
SELECT machine_id, status, health_pct FROM factory.machine
WHERE status='healthy' AND health_pct >= 90.0 LIMIT 10;
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"SELECT machine_id, status, health_pct FROM {TABLE_MACHINE} "
            f"WHERE status='healthy' AND health_pct >= 90.0 LIMIT 10"
        )
        available = cur.fetchall()

print(f'Available machines:')
for row in available:
    print(f'  {row[0]}: {row[1]}, {row[2]}%')

if any(m[0] == MACHINE_ID for m in available):
    print(f'\n{MACHINE_ID} is available.')

## Step 3 - Agent's function call 2: assign work order

The agent assigns the work order to the machine by creating an operation. The SQL:

```sql
INSERT INTO factory.operation (op_id, wo_id, machine_id, seq, status)
VALUES (gen_random_uuid()::text, 'WO-1001', 'M-204', 1, 'pending');
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"INSERT INTO {TABLE_OPERATION} (op_id, wo_id, machine_id, seq, status) "
            f"VALUES ('OP-' || gen_random_uuid()::text, %s, %s, 1, 'pending')",
            [WO_ID, MACHINE_ID]
        )

print(f'Assigned {WO_ID} to {MACHINE_ID}')

## Step 4 - Agent remembers what it did

The agent appends to the memory table. The SQL:

```sql
INSERT INTO factory.agent_memory (namespace, session_id, agent_id, role, content)
VALUES ('default', 'shift-001', 'mes_agent', 'assistant', 'Assigned WO-1001 to M-204');
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"INSERT INTO {TABLE_MEMORY} (namespace, session_id, agent_id, role, content) "
            f"VALUES (%s, %s, %s, %s, %s)",
            [NAMESPACE, SESSION_ID, "mes_agent", "assistant", f"Assigned {WO_ID} to {MACHINE_ID}"]
        )

print(f'Remembered assignment in agent_memory')

## Step 5 - Agent called again: refuses duplicate

On a second call, the agent reads memory, finds the assignment already exists, and refuses to double-assign.

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"SELECT content FROM {TABLE_MEMORY} "
            f"WHERE namespace=%s AND session_id=%s AND agent_id='mes_agent' "
            f"AND content LIKE %s",
            [NAMESPACE, SESSION_ID, f"%{WO_ID}%"]
        )
        prior_assignment = cur.fetchone()

if prior_assignment:
    print(f'Agent reads memory: "{prior_assignment[0]}"')
    print(f'REFUSE: {WO_ID} is already assigned. No double-assign.')
else:
    print(f'No prior assignment found')

## What you learned

An agent with function calling used a Lakebase table as its memory: query machines, assign work, remember the assignment. On the second call, it read its prior memory and refused to double-assign. This pattern scales to complex workflows: maintenance, quality control, scheduling - all agents consulting the same table for idempotent, coordinated work.

**Next:** Learning loop - extract, route by confidence, correct, and analyze on one table.